In [22]:
from __future__ import annotations
import os
import json
import importlib
import pprint
import time

import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

import random
import numpy as np
np.set_printoptions(4)

import torch
from datasets import load_dataset

from sal.config import Config
from core.reward_models import RLHFFlow

from utils.load_data import load_data_prm800k_hf
from utils import utils
from utils import metrics


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/kjun/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 14.95752763748169


In [33]:
# general params
config = Config()
config.agg_strategy = 'last'

level = 4 
num_trials = 5
q_idx = None

#  load data 
dataset_orig = load_data_prm800k_hf(data_dir, level, q_idx)

# run search_algo and save results
config_name = f"mcts--level-4--q-079--q31--n-4--d-20--b-80--cpuct-2"
config_name = f"mcts--level-4--e21--n-4--d-10--nb-8--cpuct-2"
# result_dir = f"results/mcts--level-{level}/prompts/q-{q_idx:03d}/{config_name}"
result_dir = f"results/mcts--level-{level}/{config_name}"
print(f"config_name = {config_name}")

importlib.reload(utils)

for trial_idx in range(num_trials):
    utils.add_completions_to_dataset(result_dir, config_name, dataset_orig, prm, trial_idx, config=config)

config_name = mcts--level-4--e21--n-4--d-10--nb-8--cpuct-2
num_questions = 128


Map:   0%|          | 0/128 [00:00<?, ? examples/s]

[0]


Computing majority & weighted predictions:   0%|          | 0/1 [00:00<?, ?it/s]

Subsample 0:   0%|          | 0/128 [00:00<?, ? examples/s]

...

Computing majority & weighted predictions: 100%|██████████| 1/1 [00:04<00:00,  4.68s/it]


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

num_questions = 128


Exception ignored in: <function Dataset.__del__ at 0x7f99ae2d2e80>
Traceback (most recent call last):
  File "/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/datasets/arrow_dataset.py", line 1415, in __del__
    def __del__(self):

KeyboardInterrupt: 


KeyboardInterrupt: 

In [35]:
importlib.reload(metrics)

def compute_stats(result_dir, config_name, num_trials):
    all_passn_correctness = []
    all_pass1b_correctness = []
    all_naive1b_correctness = []
    all_weighted1b_correctness = []
    all_maj1b_correctness = []
    
    all_pass1b_ncomps = []
    all_pass1b_lengths = []
    all_pass1b_nphases = []
    all_pass1b_ndepths = []
    
    for trial_idx in range(num_trials):
        # load data
        dataset_res = load_dataset("json", data_files = f"{result_dir}/{config_name}--trial-{trial_idx}.jsonl", split='train')
        
        passn_correctness, pass1b_correctness, naive1b_correctness, weighted1b_correctness, maj1b_correctness, \
            pass1b_ncomps, pass1b_lengths, pass1b_nphases, pass1b_ndepths = metrics.evaluate_correctness(dataset_res)
    
        # all_passn_correctness.append(passn_correctness)
        all_pass1b_correctness.append(pass1b_correctness)
        all_naive1b_correctness.append(naive1b_correctness)
        all_weighted1b_correctness.append(weighted1b_correctness)
        all_maj1b_correctness.append(maj1b_correctness)
    
        all_pass1b_ncomps.append(pass1b_ncomps)
        all_pass1b_lengths.append(pass1b_lengths)
        all_pass1b_nphases.append(pass1b_ncomps)
        all_pass1b_ndepths.append(pass1b_lengths)

    # all_passn_correctness = np.concatenate(all_passn_correctness)
    all_pass1b_correctness = np.concatenate(all_pass1b_correctness)
    all_naive1b_correctness = np.concatenate(all_naive1b_correctness)
    all_weighted1b_correctness = np.concatenate(all_weighted1b_correctness)
    all_maj1b_correctness = np.concatenate(all_maj1b_correctness)
    
    all_pass1b_ncomps = np.concatenate(all_pass1b_ncomps)
    all_pass1b_lengths = np.concatenate(all_pass1b_lengths)
    all_pass1b_nphases = np.concatenate(all_pass1b_nphases)
    all_pass1b_ndepths = np.concatenate(all_pass1b_ndepths)
    
    # print(all_pass1b_correctness.shape)
    # np.savetxt(f"{result_dir}/passn_{config_name}.txt", all_passn_correctness)
    np.savetxt(f"{result_dir}/pass1b_{config_name}.txt", all_pass1b_correctness)
    np.savetxt(f"{result_dir}/naive1b_{config_name}.txt", all_naive1b_correctness)
    np.savetxt(f"{result_dir}/weighted1b_{config_name}.txt", all_weighted1b_correctness)
    np.savetxt(f"{result_dir}/maj1b_{config_name}.txt", all_maj1b_correctness)
    
    # passn_correctness_mean = np.mean(all_passn_correctness)
    pass1b_correctness_mean = np.mean(all_pass1b_correctness)
    naive1b_correctness_mean = np.mean(all_naive1b_correctness)
    weighted1b_correctness_mean = np.mean(all_weighted1b_correctness)
    maj1b_correctness_mean = np.mean(all_maj1b_correctness)

    num_questions = len(dataset_res)
    # passn_correctness_std = np.std(all_passn_correctness, ddof=1)/np.sqrt(num_trials*num_questions)  
    pass1b_correctness_std = np.std(all_pass1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)
    naive1b_correctness_std = np.std(all_naive1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)
    weighted1b_correctness_std = np.std(all_weighted1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)
    maj1b_correctness_std = np.std(all_maj1b_correctness, ddof=1)/np.sqrt(num_trials*num_questions)

    pass1b_ncomps_mean = np.mean(all_pass1b_ncomps)
    pass1b_lengths_mean = np.mean(all_pass1b_lengths)
    pass1b_ncomps_std = np.std(all_pass1b_ncomps, ddof=1)/np.sqrt(num_trials*num_questions)
    pass1b_lengths_std = np.std(all_pass1b_lengths, ddof=1)/np.sqrt(num_trials*num_questions)

    pass1b_nphases_mean = np.mean(all_pass1b_nphases)
    pass1b_ndepths_mean = np.mean(all_pass1b_ndepths)
    pass1b_nphases_std = np.std(all_pass1b_nphases, ddof=1)/np.sqrt(num_trials*num_questions)
    pass1b_ndepths_std = np.std(all_pass1b_ndepths, ddof=1)/np.sqrt(num_trials*num_questions)

    print(
        f"{pass1b_correctness_mean:0.4f} (\u00B1{pass1b_correctness_std:0.4f}), "
        f"{naive1b_correctness_mean:0.4f} (\u00B1{naive1b_correctness_std:0.4f}),"
        f"{weighted1b_correctness_mean:0.4f} (\u00B1{weighted1b_correctness_std:0.4f}), "
        f"{maj1b_correctness_mean:0.4f} (\u00B1{maj1b_correctness_std:0.4f}), "
        f"{pass1b_ncomps_mean:0.1f}    (\u00B1{pass1b_ncomps_std:0.1f}), "
        f"{pass1b_lengths_mean:0.1f}    (\u00B1{pass1b_lengths_std:0.1f}), "
        f"{pass1b_nphases_mean:0.1f}    (\u00B1{pass1b_nphases_std:0.1f}), "
        f"{pass1b_ndepths_mean:0.1f}    (\u00B1{pass1b_ndepths_std:0.1f})"
    )

# config_name = f"mcts--level-4--q-079--q31--n-4--d-20--b-80--cpuct-2"
config_name = f"mcts--level-4--e21--n-4--d-10--nb-8--cpuct-2"
# result_dir = f"results/mcts--level-{level}/prompts/q-{q_idx:03d}/{config_name}"
result_dir = f"results/mcts--level-{level}/{config_name}"
print(f"config_name = {config_name}")

compute_stats(result_dir, config_name, num_trials=1)

config_name = mcts--level-4--e21--n-4--d-10--nb-8--cpuct-2


/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/u20/tnguyen9210/micromamba/envs/py311/lib/python3.11/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


0.5859 (±0.0437), 0.4297 (±0.0439),0.4609 (±0.0442), 0.4375 (±0.0440), 10.3    (±0.8), nan    (±nan), 10.3    (±0.8), nan    (±nan)


In [25]:
tmp = np.mean([7, 8, 8, 8, 2, 2, 5, 7, 7, 8, 6, 7, 11, 11, 9, 18, 18, 6, 5])
print(tmp)

8.052631578947368
